# 04 - Forensic Error Analysis & Case Triage
### Fraud Risk Intelligence & Detection Platform
This notebook covers:
1. Deep-dive into False Positives (why legitimate transactions trigger alerts).
2. Deep-dive into False Negatives (why subtle frauds slip through).
3. Amount tier segment analysis.
4. Multi-signal risk decisioning (`ALLOW`, `FLAG_CHALLENGE`, `BLOCK_AND_INVESTIGATE`).
5. Priority investigation queue ranking based on risk score and monetary exposure.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.utils import load_artifact
from src.risk import RiskEngine, InvestigationTriage
print("Imports loaded successfully.")


## 1. Dissecting Model Errors on Test Data


In [ ]:
test_df = pd.read_parquet("data/processed/test.parquet")
pipeline = load_artifact("artifacts/models/feature_pipeline.joblib")
xgb_model = load_artifact("artifacts/models/xgboost_model.joblib")

X_test = pipeline.transform(test_df)
y_test = test_df["Class"].values.astype(int)
probs = xgb_model.predict_proba(X_test)
preds = (probs >= xgb_model.best_threshold).astype(int)

error_df = test_df.copy()
error_df["prob"] = probs
error_df["pred"] = preds
error_df["error_type"] = "TN"
error_df.loc[(y_test == 0) & (preds == 1), "error_type"] = "FP (False Alarm)"
error_df.loc[(y_test == 1) & (preds == 0), "error_type"] = "FN (Missed Fraud)"
error_df.loc[(y_test == 1) & (preds == 1), "error_type"] = "TP (Caught Fraud)"

print("Error Breakdown:")
print(error_df["error_type"].value_counts())


### False Positive Analysis (Why were they flagged?)
Let's contrast False Positives with True Negatives on Transaction Amount and PCA risk features.


In [ ]:
fp_rows = error_df[error_df["error_type"] == "FP (False Alarm)"]
tn_rows = error_df[error_df["error_type"] == "TN"]

print(f"FP Mean Amount: ${fp_rows['Amount'].mean():.2f} vs TN Mean Amount: ${tn_rows['Amount'].mean():.2f}")
print(f"FP Max Amount:  ${fp_rows['Amount'].max():.2f} vs TN Max Amount:  ${tn_rows['Amount'].max():.2f}")
print("\nTop 5 False Positive Amounts:")
print(fp_rows[['Time', 'Amount', 'prob', 'V14', 'V4']].head(5))


## 2. Amount Tier Segment Performance


In [ ]:
error_df["amount_tier"] = pd.cut(
    error_df["Amount"],
    bins=[-0.1, 5.0, 500.0, 100000.0],
    labels=["Micro (<$5)", "Medium ($5-$500)", "High (>$500)"],
)
tier_perf = error_df.groupby("amount_tier", observed=False).agg(
    total_tx=("Class", "count"),
    fraud_count=("Class", "sum"),
    caught_fraud=("pred", lambda p: int(((error_df.loc[p.index, "Class"] == 1) & (p == 1)).sum())),
    false_positives=("pred", lambda p: int(((error_df.loc[p.index, "Class"] == 0) & (p == 1)).sum())),
)
tier_perf["recall"] = tier_perf["caught_fraud"] / tier_perf["fraud_count"].replace(0, np.nan)
print(tier_perf.to_string())


## 3. End-to-End Multi-Signal Decision Engine
Combines Supervised Probability (0-100) + Anomaly Score (0-100) into a transparent risk score and 3-tier routing.


In [ ]:
risk_engine = RiskEngine()

# Test 1: Clean transaction
clean_decision = risk_engine.evaluate(
    ml_probability=0.01,
    anomaly_score=0.10,
    amount=45.0,
)
print(f"Clean Transaction -> Score: {clean_decision.risk_score:.1f} | Tier: {clean_decision.risk_level} | Action: {clean_decision.action}")

# Test 2: Borderline high anomaly transaction
anom_decision = risk_engine.evaluate(
    ml_probability=0.45,
    anomaly_score=0.85,
    amount=2500.0,
)
print(f"Anomalous Tx      -> Score: {anom_decision.risk_score:.1f} | Tier: {anom_decision.risk_level} | Action: {anom_decision.action}")
print(f"  Reasons: {anom_decision.primary_reasons}")
print(f"  Priority Exposure Score: {anom_decision.priority_score:.1f}")


## 4. Priority Investigation Queue Ranking
Ranks cases by monetary exposure at risk ($	ext{Priority Score} = 	ext{Risk Score} \times \ln(1 + \text{Amount})$).


In [ ]:
queue_sample = error_df[error_df["pred"] == 1].head(15).copy()
queue_sample["risk_score"] = queue_sample["prob"] * 100.0
ranked_cases = InvestigationTriage.rank_cases(queue_sample, top_n=5)

print("Top 5 Cases for Human Reviewer Queue:")
print(ranked_cases[["Time", "Amount", "prob", "risk_score", "priority_score", "Class"]].round(2))
